# SCFP V2 — train từ dataset gốc trên Kaggle

**3 cell:** cấu hình → git clone/setup → train. Bật **GPU + Internet**, attach **dataset ICFG-PEDES gốc** (`ICFG-PEDES.json` và `imgs/`), rồi **Run All**. Code được clone từ branch `feat/scfp-v2` của repo ở cell đầu; branch phải chứa bản launcher mới.

Thiết lập IRRA: CLIP ViT-B/16, ảnh 384×128, Adam, LR 1e-5, batch 64, 60 epoch, augmentation và SDM + MLM + ID; cosine scheduler, warmup 5 epoch. Cấu hình được tạo cho từng run; `configs.yaml` là **file output**, không phải dữ liệu đầu vào. Các file `processed_data/*.pkl` không cần cho pipeline này.

Theo IRRA gốc: dùng full train, theo dõi official-test R1 mỗi epoch, lưu best/last; đây không phải validation độc lập. Cuối run tính đủ R1/R5/R10/mAP/mINP. Chọn `EXPERIMENT="baseline"` để chạy IRRA đối chứng cùng thiết lập. Logs/checkpoints nằm trong Kaggle Output. Notebook không tự chạy unit tests hay smoke tests.

In [ ]:
EXPERIMENT = "v2"  # baseline | slots | frequency | v2
EPOCHS = 60
BATCH_SIZE = 64
SEED = 1
BACKBONE_LR = 1e-5
SCFP_LR = 1e-4
NUM_WORKERS = 4

DATASET_DIR = "/kaggle/input/datasets/tphu25/icfg-pedes-dataset/ICFG-PEDES"
REPO_URL = "https://github.com/thanhphu25/IRRA.git"
BRANCH = "feat/scfp-v2"
RESUME_CHECKPOINT = None  # last.pth nếu tiếp tục một run đã lưu
OUTPUT_ROOT = "/kaggle/working/SCFP_V2_train"
EVALUATE_AFTER_TRAIN = True


In [ ]:
import os, sys, subprocess, tempfile
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO = Path(tempfile.mkdtemp(prefix="IRRA_scfp_v2_train_", dir="/kaggle/working")).resolve()
subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
required = ("experiments/scfp/kaggle_train.py", "experiments/scfp/requirements-train.txt")
if not all((REPO / name).is_file() for name in required):
    raise RuntimeError(f"Branch {BRANCH} trên GitHub chưa có code notebook train mới. Cần push bản cập nhật trước khi Run All.")
os.chdir(REPO)
sys.path.insert(0, str(REPO))
# Dùng đúng source vừa clone nếu chạy lại notebook trong cùng một kernel.
cached_launcher = sys.modules.get("experiments.scfp.kaggle_train")
if cached_launcher is not None:
    previous_root = Path(cached_launcher.__file__).resolve().parents[2]
    for name, module in list(sys.modules.items()):
        filename = getattr(module, "__file__", None)
        if filename and Path(filename).resolve().is_relative_to(previous_root):
            sys.modules.pop(name, None)
    for name in ("experiments.scfp", "experiments"):
        sys.modules.pop(name, None)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "experiments/scfp/requirements-train.txt"], check=True)
import torch
assert torch.cuda.is_available(), "Bật GPU trong Kaggle Settings"
assert torch.cuda.device_count() == 1, "Notebook dùng GPU 0"
commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("GPU:", torch.cuda.get_device_name(0), "| torch:", torch.__version__, "| commit:", commit)


In [ ]:
from experiments.scfp.kaggle_train import prepare_training, run_training

plan = prepare_training(
    dataset_dir=DATASET_DIR, output_root=OUTPUT_ROOT,
    experiment=EXPERIMENT, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED,
    backbone_lr=BACKBONE_LR, scfp_lr=SCFP_LR, num_workers=NUM_WORKERS,
    resume_checkpoint=RESUME_CHECKPOINT,
)
RUN_DIR = run_training(plan, evaluate=EVALUATE_AFTER_TRAIN)
